# Solar Filament Segmentation — v13 reproducible inference

Loads `weights/best.pt` and recreates v13 inference: 1024×1024 input, no TTA, fixed component thresholds, and 2048×2048 RLE output. This notebook does not train the model or retune thresholds.

Run from the repository root. Set `filament.CFG['test_dir']` if the test data is not at the default Kaggle dataset path.

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, REPO_ROOT)
import filament as F
import torch

F.CFG.update(
    size=1024,
    ckpt=os.path.join(REPO_ROOT, 'weights', 'best.pt'),
    t_lo=0.35,
    t_hi=0.50,
    min_meanp=0.30,
    min_area=50,
    close_k=5,
    out_size=2048,
)

print('device:', F.dev)
if F.dev == 'cuda':
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))

## 1. Check checkpoint

The v13 checkpoint is supplied in `weights/best.pt`; this reproduction notebook never retrains.

In [ ]:
if not os.path.isfile(F.CFG['ckpt']):
    raise FileNotFoundError(f"v13 checkpoint not found: {F.CFG['ckpt']}")
print('using v13 checkpoint:', F.CFG['ckpt'])

## 2. Load model

In [ ]:
model = F.build(pretrained=False).to(F.dev)
model.load_state_dict(torch.load(F.CFG['ckpt'], map_location=F.dev))
model.eval()
print('loaded', F.CFG['ckpt'])

## 3. Fixed v13 inference settings

The submitted v13 predictions used a single 1024×1024 forward pass (no TTA) and the fixed thresholds below. Do not retune them when reproducing v13.

In [ ]:
print({k: F.CFG[k] for k in ('size', 't_lo', 't_hi', 'min_meanp', 'min_area', 'close_k', 'out_size')})

## 4. Predict test set

In [ ]:
F.predict('submission_v13.csv', tta=False)

## 5. Validate submission format

In [ ]:
import glob, pandas as pd, numpy as np, re
from pycocotools import mask as mu

df = pd.read_csv('submission_v13.csv')
test_stems = {F.stem(p) for p in glob.glob(F.CFG['test_dir'] + '/*')}
pred_stems = set(df['filament_id'].str.rsplit('_', n=1).str[0])

checks = {}
checks['header_ok']           = list(df.columns) == ['filament_id', 'segmentation_rle']
checks['ids_unique']          = df['filament_id'].is_unique
checks['ids_match_pattern']   = df['filament_id'].str.match(r'^[A-Za-z0-9]+_\d+$').all()
checks['predicted_stems_in_test'] = pred_stems.issubset(test_stems)
checks['no_empty_rles']       = df['segmentation_rle'].notna().all() and df['segmentation_rle'].str.len().gt(0).all()
checks['no_empty_masks']      = True
checks['decode_2048_ok']      = True

for _, row in df.iterrows():
    if pd.isna(row['segmentation_rle']) or not row['segmentation_rle']:
        checks['no_empty_masks'] = False
        continue
    rle_obj = {'size': [2048, 2048], 'counts': row['segmentation_rle'].encode()}
    try:
        m = mu.decode(rle_obj)
        if m.shape != (2048, 2048) or m.sum() == 0: checks['no_empty_masks'] = False
    except Exception:
        checks['decode_2048_ok'] = False

for k, v in checks.items():
    print(f'  {k:<28} {v}')
missing_stems = sorted(test_stems - pred_stems)
print(f'\n{len(df)} rows, {len(pred_stems)} of {len(test_stems)} test images have predictions')
if missing_stems:
    print('No predicted instances for:', ', '.join(missing_stems))
if not all(checks.values()):
    raise ValueError('Submission validation failed; do not submit this CSV.')